# Model 2 · Experiments: Pipeline 2 selection and whole-app test

**Pipeline 2** ranks the ~100 labeled library dishes for the cook's request with a Hugging Face
`feature-extraction` pipeline (sentence embeddings + cosine similarity, plus an exact-match bonus).
It works well without training, so it is **not fine-tuned**: we test three pre-trained encoders on our
own queries and pick one on accuracy and run time.

Steps in this notebook
1. Test three pre-trained encoders on our own queries (600 development, 200 test, 24 paraphrase).
2. Pick the encoder for Pipeline 2.
3. Test the whole app (Pipeline 1 fine-tuned + Pipeline 2 + rules) on 20 dinner requests.
4. Write `Experimental_results.xlsx`.

## 0 · Setup

In [1]:
# Runs in Colab ("Restart & Run All") and locally. Clones the project if it is not already here.
import gc, json, os, random, subprocess, sys, time, warnings
from pathlib import Path

REPO_URL = "https://github.com/Tannedfish/same-table.git"
here = Path.cwd()
if not any((p / "same_table").is_dir() for p in (here, here.parent)):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "same-table"], check=True)
    os.chdir("same-table")
ROOT = Path.cwd() if (Path.cwd() / "same_table").is_dir() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

try:
    import datasets, sentence_transformers, sklearn, transformers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "datasets",
                    "sentence-transformers", "scikit-learn", "accelerate", "openpyxl"], check=True)

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)

DATA, OUT = ROOT / "data" / "train", ROOT / "artifacts"
OUT.mkdir(exist_ok=True)
HF_USER = "Tannedfish"            # Hugging Face username that owns the fine-tuned model
SEED = 5240

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu | torch 2.14.1+cpu


## 1 · Data for Pipeline 2

* **Literal queries:** each names an ingredient or method of one target dish ("雪櫃得返菠菜，點算").
  A query is right when that dish is in the top 5 (the menu shows 6–8 dishes, so top-5 is what matters).
  Several dishes share ingredients, which caps top-1 accuracy.
  **600 development queries** set one number (the hybrid weight, section 2); **200 test queries**, written
  with different sentence templates, are used only for the final comparison.
* **Paraphrase queries (24, hand-written, test only):** name a food family or use English ("禽肉", "tofu").
  Scored by precision in the top 5 against a rule on the dish name. Small set: a secondary signal.

In [2]:
from dataclasses import replace
from same_table.config import BGE_QUERY_PREFIX, Settings
from same_table.library import load_library
from same_table.retrieval import CharTfidf, Retriever, hybrid

if not (DATA / "p2_dev.jsonl").exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "make_training_data.py")], check=True)

base_settings = replace(Settings.load(), offline=False)
library = load_library(base_settings.library_path)
doc_ids = [d.id for d in library]
dev, test, paraphrase = (read_jsonl(DATA / f) for f in ("p2_dev.jsonl", "p2_test.jsonl", "p2_paraphrase_test.jsonl"))
print(f"library={len(doc_ids)} dishes | development queries={len(dev)} | test queries={len(test)} literal + {len(paraphrase)} paraphrase")
pd.DataFrame(test[:3] + [{"query": r["query"], "dish_id": f"{len(r['relevant_ids'])} relevant dishes"} for r in paraphrase[:3]])

library=100 dishes | development queries=600 | test queries=200 literal + 24 paraphrase


,query,dish_id
0,青椒鹹豬肉可以點整,r00622
1,想要一道炒嘅肉，材料有鹹豬肉,r00622
2,街市買到靚紅蘿蔔，想快炒,r00207
3,想食禽肉,10 relevant dishes
4,紅肉嘅餸,23 relevant dishes
5,想食綠葉菜,18 relevant dishes


## 2 · Three pre-trained encoders

| | Model | Language | Pooling |
|---|---|---|---|
| Model 1 | `BAAI/bge-small-zh-v1.5` | Chinese | CLS |
| Model 2 | `shibing624/text2vec-base-chinese` | Chinese | mean |
| Model 3 | `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` | 50+ languages | mean |

Each runs through the app's own `Retriever` class (a Hugging Face `feature-extraction` pipeline), so the
experiment uses exactly the code path of the app.

**Hybrid score** = cosine similarity + *w* × character TF-IDF. The encoder understands meaning; the
character match rewards a literal ingredient hit. *w* is the only number set from data: for each encoder
we take the **smallest *w* whose development accuracy is within 1 point of the best**, which keeps as
much of the encoder's meaning as possible without losing literal accuracy.

In [3]:
# model id -> (pooling, query prefix recommended by the model card)
P2_CANDIDATES = {
    "BAAI/bge-small-zh-v1.5": ("cls", BGE_QUERY_PREFIX),
    "shibing624/text2vec-base-chinese": ("mean", ""),
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": ("mean", ""),
}
WEIGHTS = [0.5, 1, 2, 3, 5, 8]

def score_matrix(score_fn, rows):
    """One row per query, one column per library dish (score_fn returns {dish_id: score})."""
    out = np.zeros((len(rows), len(doc_ids)), dtype=np.float32)
    for n, r in enumerate(rows):
        scores = score_fn(r["query"])
        out[n] = [scores[i] for i in doc_ids]
    return out

def ranks_of(S, rows):
    return np.array([int(np.where(np.array(doc_ids)[np.argsort(-s)] == r["dish_id"])[0][0]) + 1 for s, r in zip(S, rows)])

def top5(S, rows):
    return float((ranks_of(S, rows) <= 5).mean())

def precision_at_5(S, rows):
    return np.array([np.mean([doc_ids[j] in set(r["relevant_ids"]) for j in np.argsort(-s)[:5]]) for s, r in zip(S, rows)])

IS_EN = np.array([r["query"].isascii() for r in paraphrase])
def summarise(S_test, S_para):
    ranks, p5 = ranks_of(S_test, test), precision_at_5(S_para, paraphrase)
    return {"top5_accuracy": float((ranks <= 5).mean()), "top1_accuracy": float((ranks <= 1).mean()),
            "paraphrase_p@5": float(p5.mean()), "paraphrase_p@5_english": float(p5[IS_EN].mean())}, ranks

tfidf = CharTfidf({d.id: d.search_text() for d in library})
t0 = time.perf_counter(); LEX_TEST = score_matrix(tfidf.scores, test); lex_seconds = time.perf_counter() - t0
LEX_DEV, LEX_PARA = score_matrix(tfidf.scores, dev), score_matrix(tfidf.scores, paraphrase)

p2_results, p2_ranks, weight_curves = {}, {}, {}
BASELINE = "character TF-IDF (no model)"
p2_results[BASELINE], p2_ranks[BASELINE] = summarise(LEX_TEST, LEX_PARA)
p2_results[BASELINE].update({"dev_top5_accuracy": top5(LEX_DEV, dev), "seconds_per_100_queries": 100 * lex_seconds / len(test), "params_M": 0.0})

for model_id, (pooling, prefix) in P2_CANDIDATES.items():
    settings = replace(base_settings, p2_model_id=model_id, p2_pooling=pooling, p2_query_prefix=prefix, p2_hybrid_weight=0.0)
    retriever = Retriever(settings, library).load()          # weight 0 -> rank() returns the plain cosine similarity
    assert retriever.available, retriever.error
    retriever.rank("warm up")
    t0 = time.perf_counter(); D_test = score_matrix(retriever.rank, test); seconds = time.perf_counter() - t0
    D_dev, D_para = score_matrix(retriever.rank, dev), score_matrix(retriever.rank, paraphrase)

    curve = {w: top5(hybrid(D_dev, LEX_DEV, w), dev) for w in WEIGHTS}                 # development queries only
    weight = min(w for w in WEIGHTS if curve[w] >= max(curve.values()) - 0.01)
    weight_curves[model_id] = curve
    p2_results[model_id], p2_ranks[model_id] = summarise(hybrid(D_test, LEX_TEST, weight), hybrid(D_para, LEX_PARA, weight))
    dense_only, _ = summarise(D_test, D_para)
    p2_results[model_id].update({
        "hybrid_weight": weight, "dev_top5_accuracy": curve[weight], "top5_accuracy_dense_only": dense_only["top5_accuracy"],
        "paraphrase_p@5_dense_only": dense_only["paraphrase_p@5"], "seconds_per_100_queries": 100 * seconds / len(test),
        "params_M": sum(p.numel() for p in retriever._pipe.model.parameters()) / 1e6})
    print(model_id, {k: round(v, 3) for k, v in p2_results[model_id].items()})
    del retriever; gc.collect()

pd.DataFrame(weight_curves).T.rename_axis("development top-5 accuracy by weight w").round(3)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

Loading weights:  17%|█▋        | 12/71 [00:00<00:00, 61.01it/s]

Loading weights:  27%|██▋       | 19/71 [00:00<00:01, 43.59it/s]

Loading weights:  34%|███▍      | 24/71 [00:00<00:01, 34.50it/s]

Loading weights:  39%|███▉      | 28/71 [00:00<00:01, 26.36it/s]

...

Loading weights:  93%|█████████▎| 186/199 [00:06<00:00, 49.47it/s]

Loading weights:  97%|█████████▋| 194/199 [00:06<00:00, 53.29it/s]

Loading weights: 100%|██████████| 199/199 [00:06<00:00, 30.52it/s]

sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 {'top5_accuracy': 0.895, 'top1_accuracy': 0.6, 'paraphrase_p@5': 0.6, 'paraphrase_p@5_english': 0.725, 'hybrid_weight': 5, 'dev_top5_accuracy': 0.892, 'top5_accuracy_dense_only': 0.3, 'paraphrase_p@5_dense_only': 0.608, 'seconds_per_100_queries': 2.719, 'params_M': 117.654}


,0.5,1.0,2.0,3.0,5.0,8.0
development top-5 accuracy by weight w,,,,,,
BAAI/bge-small-zh-v1.5,0.833,0.863,0.897,0.905,0.910,0.910
shibing624/text2vec-base-chinese,0.828,0.883,0.903,0.907,0.908,0.907
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2,0.672,0.798,0.865,0.888,0.892,0.902


## 3 · Model selection: accuracy and run time

Rule (same as for Pipeline 1): highest **top-5 accuracy on the 200 literal test queries**; models within
one standard error of the best are treated as tied and separated by run time. The character TF-IDF row is
the no-model baseline.

In [4]:
p2_table = pd.DataFrame(p2_results).T[["top5_accuracy", "top1_accuracy", "paraphrase_p@5", "paraphrase_p@5_english",
                                       "hybrid_weight", "top5_accuracy_dense_only", "seconds_per_100_queries", "params_M"]].round(3)
candidates = p2_table.loc[list(P2_CANDIDATES)]
best = candidates.top5_accuracy.max()
standard_error = float(np.sqrt(best * (1 - best) / len(test)))
close = candidates[candidates.top5_accuracy >= best - standard_error]
P2_CHOSEN = close.seconds_per_100_queries.idxmin()
print(f"best top-5 accuracy {best:.3f}, standard error {standard_error:.3f} -> tied models: {list(close.index)}")
P2_WEIGHT = float(p2_results[P2_CHOSEN]["hybrid_weight"])
print("Picked:", P2_CHOSEN, "| hybrid weight", P2_WEIGHT)
p2_table

best top-5 accuracy 0.895, standard error 0.022 -> tied models: ['BAAI/bge-small-zh-v1.5', 'shibing624/text2vec-base-chinese', 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2']
Picked: BAAI/bge-small-zh-v1.5 | hybrid weight 3.0


,top5_accuracy,top1_accuracy,paraphrase_p@5,paraphrase_p@5_english,hybrid_weight,top5_accuracy_dense_only,seconds_per_100_queries,params_M
character TF-IDF (no model),0.890,0.640,0.475,0.600,NaN,NaN,0.026,0.000
BAAI/bge-small-zh-v1.5,0.895,0.655,0.542,0.600,3.0,0.720,2.262,23.954
shibing624/text2vec-base-chinese,0.895,0.620,0.583,0.600,2.0,0.595,5.891,102.268
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2,0.895,0.600,0.600,0.725,5.0,0.300,2.719,117.654


In [5]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
short = [i.split("/")[-1].replace(" (no model)", "").replace("paraphrase-multilingual-", "multilingual-") for i in p2_table.index]
colors = ["#1F4D3A" if i == P2_CHOSEN else "#9DB8A9" for i in p2_table.index]
for a, col, title in ((ax[0], "top5_accuracy", "Top-5 accuracy, 200 literal test queries"),
                      (ax[1], "paraphrase_p@5", "Precision@5, 24 paraphrase queries")):
    a.bar(short, p2_table[col], color=colors); a.set_ylim(0, 1.08); a.set_title(title, fontsize=10)
    for x, v in enumerate(p2_table[col]): a.text(x, v + 0.015, f"{v:.3f}", ha="center", fontsize=8)
ax[2].bar(short, p2_table.seconds_per_100_queries, color=colors); ax[2].set_title("Run time: seconds per 100 queries (CPU)", fontsize=10)
for a in ax: a.tick_params(axis="x", labelsize=7, rotation=12)
plt.tight_layout(); plt.savefig(OUT / "p2_comparison.png", dpi=150); plt.show()

## 4 · Whole-app test

The app = Pipeline 1 (our fine-tuned tagger + keyword lexicon) → Pipeline 2 (picked encoder) → rules.
It is loaded here through the same classes `app.py` uses. Twenty dinner requests are judged by objective
checks; a request is **Correct** only if every check passes:

* the tags equal the expected tags;
* the number of dishes equals the dish-count table for that headcount;
* no dish repeats, and every requested cooking method appears on the menu;
* a soup is on the menu when one is asked for (or the table has 4+ dishes);
* "within 20 min" menus contain only dishes of 20 minutes or less (unless the app says it relaxed the rule);
* "more veg" menus have more vegetable dishes than meat dishes, and the reverse for "more meat".

In [6]:
from datetime import date
from same_table.planner import plan_tonight
from same_table.rules import dish_count, seed_log
from same_table.tagging import ALL_TAGS, Tagger

local_tagger = OUT / "same-table-tagger"
P1_MODEL = str(local_tagger) if local_tagger.exists() else f"{HF_USER}/same-table-tagger"
pooling, prefix = P2_CANDIDATES[P2_CHOSEN]
settings = replace(base_settings, p1_model_id=P1_MODEL, p1_task="text-classification", p1_threshold=0.5,
                   p2_model_id=P2_CHOSEN, p2_pooling=pooling, p2_query_prefix=prefix, p2_hybrid_weight=P2_WEIGHT)
tagger, retriever = Tagger(settings).load(), Retriever(settings, library).load()
assert tagger.available and retriever.available, (tagger.error, retriever.error)
print("Pipeline 1:", P1_MODEL, "\nPipeline 2:", P2_CHOSEN, "with hybrid weight", P2_WEIGHT)

# the deployed Retriever must give the ranks the experiment reported
app_ranks = ranks_of(score_matrix(retriever.rank, test), test)
assert (app_ranks == p2_ranks[P2_CHOSEN]).all(), "app and experiment disagree"
print("app ranks equal experiment ranks on all", len(test), "test queries")

def tag_set(tags):
    found = set(tags.methods) | ({tags.balance} - {"balanced"})
    return found | {name for name in ("less_sweet", "quick") if getattr(tags, name)}

# Pipeline 1 as deployed (model + keyword lexicon) on the 108 hand-written sentences
p1_test = read_jsonl(DATA / "p1_test.jsonl")
t0 = time.perf_counter()
app_tags = [tag_set(tagger.tag(r["text"])) for r in p1_test]
p1_app_seconds = time.perf_counter() - t0
p1_app_accuracy = float(np.mean([found == set(r["tags"]) for found, r in zip(app_tags, p1_test)]))
print(f"Pipeline 1 in the app: {p1_app_accuracy:.3f} accuracy on {len(p1_test)} hand-written sentences "
      f"({100 * p1_app_seconds / len(p1_test):.1f} s per 100 requests)")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   6%|▌         | 6/105 [00:00<00:03, 29.09it/s]

Loading weights:  19%|█▉        | 20/105 [00:00<00:02, 36.30it/s]

Loading weights:  23%|██▎       | 24/105 [00:00<00:03, 26.32it/s]

Loading weights:  29%|██▊       | 30/105 [00:01<00:02, 25.61it/s]

...

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 6209.90it/s]

Pipeline 1: /home/claude/same_table_app/artifacts/same-table-tagger 
Pipeline 2: BAAI/bge-small-zh-v1.5 with hybrid weight 3.0


app ranks equal experiment ranks on all 200 test queries


Pipeline 1 in the app: 0.889 accuracy on 108 hand-written sentences (11.2 s per 100 requests)


In [7]:
# (people, sentence typed by the cook, chips pressed, expected tags)
APP_TESTS = [
    (4, "阿爸牙唔好，想食啲軟熟嘅蒸餸，唔好太甜", [], {"steam", "less_sweet"}),
    (2, "今晚好趕，求其炒兩個餸就算", [], {"stir", "quick"}),
    (6, "想煲湯，另外加個白灼菜", [], {"soup", "blanch"}),
    (5, "屋企有三個男仔，肉要多", [], {"more_meat"}),
    (3, "最近食得好油，想清清地", [], {"more_veg"}),
    (4, "", ["steam", "soup"], {"steam", "soup"}),
    (4, "", [], set()),
    (1, "七點先放工，要十五分鐘搞得掂", [], {"quick"}),
    (6, "婆婆糖尿，醬汁唔可以甜", ["more_veg"], {"less_sweet", "more_veg"}),
    (3, "今晚想吃清蒸的，配一個燙青菜", [], {"steam", "blanch"}),
    (4, "quick stir fry with lots of veggies", [], {"stir", "quick", "more_veg"}),
    (5, "we want more meat tonight and a soup", [], {"more_meat", "soup"}),
    (2, "落雨天想飲啖熱湯", [], {"soup"}),
    (4, "想食得有鑊氣，炒牛肉都好", [], {"stir"}),
    (6, "蒸魚，炒菜，再加個湯", [], {"steam", "stir", "soup"}),
    (3, "十五分鐘煮好，灼個菜蒸條魚", [], {"quick", "blanch", "steam"}),
    (4, "多菜少肉，唔好甜", ["quick"], {"more_veg", "less_sweet", "quick"}),
    (2, "隨便啦", ["blanch"], {"blanch"}),
    (5, "長輩血糖高，不要甜的", [], {"less_sweet"}),
    (4, "雪櫃有雞胸同菠菜", [], set()),
]
today = date(2026, 10, 1)        # fixed date so the 14-day demo log, and therefore the menus, are reproducible

def judge(people, text, chips, expected):
    res = plan_tonight(people=people, text=text, chips=chips, library=library, tagger=tagger, retriever=retriever,
                       log=seed_log(library, today), today=today)
    dishes = [library.get(i) for i in res.plan.dish_ids]
    found = tag_set(res.tags)
    cats = [d.category for d in dishes]
    checks = {
        "tags": found == expected,
        "dish count": len(dishes) == dish_count(people),
        "no repeats": len(set(res.plan.dish_ids)) == len(dishes),
        "methods on menu": all(any(d.method == m for d in dishes) for m in res.tags.methods),
        "soup": ("soup" in cats) if ("soup" in expected or len(dishes) >= 4) else True,
        "20 min": all(d.is_quick for d in dishes) if (res.tags.quick and "time_relaxed" not in res.plan.notes) else True,
        "balance": (cats.count("veg") > cats.count("meat")) if res.tags.balance == "more_veg"
                   else (cats.count("meat") > cats.count("veg")) if res.tags.balance == "more_meat" else True,
    }
    failed = [k for k, ok in checks.items() if not ok]
    shown = text if text else "(empty box)"
    return {"people": people, "input": shown + (f"  [chips: {', '.join(chips)}]" if chips else ""),
            "expected tags": ", ".join(sorted(expected)) or "(none)", "tags found": ", ".join(sorted(found)) or "(none)",
            "menu": " / ".join(d.name for d in dishes), "notes": ", ".join(res.plan.notes),
            "result": "Correct" if not failed else "Incorrect", "failed checks": ", ".join(failed)}

t0 = time.perf_counter()
app_rows = [judge(*case) for case in APP_TESTS]
app_seconds = time.perf_counter() - t0
app_table = pd.DataFrame(app_rows)
app_accuracy = float((app_table.result == "Correct").mean())
print(f"App accuracy: {(app_table.result == 'Correct').sum()} out of {len(app_table)} = {app_accuracy:.2f}  "
      f"({app_seconds / len(app_table):.2f} s per request on {DEVICE})")
app_table[["people", "input", "expected tags", "tags found", "result", "failed checks"]]

App accuracy: 18 out of 20 = 0.90  (0.05 s per request on cpu)


,people,input,expected tags,tags found,result,failed checks
0,4,阿爸牙唔好，想食啲軟熟嘅蒸餸，唔好太甜,"less_sweet, steam","less_sweet, steam",Correct,
1,2,今晚好趕，求其炒兩個餸就算,"quick, stir","quick, stir",Correct,
2,6,想煲湯，另外加個白灼菜,"blanch, soup","blanch, soup",Correct,
3,5,屋企有三個男仔，肉要多,more_meat,more_meat,Correct,
4,3,最近食得好油，想清清地,more_veg,more_veg,Correct,
5,4,"(empty box) [chips: steam, soup]","soup, steam","soup, steam",Correct,
6,4,(empty box),(none),(none),Correct,
7,1,七點先放工，要十五分鐘搞得掂,quick,quick,Correct,
8,6,婆婆糖尿，醬汁唔可以甜 [chips: more_veg],"less_sweet, more_veg","less_sweet, more_veg",Correct,
9,3,今晚想吃清蒸的，配一個燙青菜,"blanch, steam","blanch, steam",Correct,


In [8]:
app_table[["input", "menu", "notes"]]

,input,menu,notes
0,阿爸牙唔好，想食啲軟熟嘅蒸餸，唔好太甜,不會失敗的蒸蛋 / 清蒸山藥｜三鮮肉丸 / 蝦仁蒸蛋豆腐 / 甜豌豆炒菇 / 蒜炒小白菜/快炒/5分鐘/簡單快速 / 客家鹹湯圓的“炒料”,
1,今晚好趕，求其炒兩個餸就算,好市多牛三寶炒蒜苗 / 菠菜炒鴻禧菇 / 蒜炒小白菜/快炒/5分鐘/簡單快速,
2,想煲湯，另外加個白灼菜,白切雞 / 『涼拌皮蛋』－清爽、酸辣、開胃。 / 涼拌魚皮 / 芹菜炒花枝 / 家常小菜-涼拌干絲 / 《料理簡單做》涼拌萵筍/菜心 / 涼拌萵筍 ( A菜心 ) / 客家...,
3,屋企有三個男仔，肉要多,好市多牛三寶炒蒜苗 / 十三香炒回鍋肉 / 青椒炒肉絲 - 家常料理 / 菠菜炒鴻禧菇 / 免炒無油無糖自製棗泥 / 素食版的粄條😋 蔬炒粄條🤤 / 薑絲石斑魚蛤蜊湯,
4,最近食得好油，想清清地,『涼拌皮蛋』－清爽、酸辣、開胃。 / 清炒小黃瓜 / 蒜炒菜心 / 山藥排骨湯,
5,"(empty box) [chips: steam, soup]",不會失敗的蒸蛋 / 清蒸山藥｜三鮮肉丸 / 清蒸石斑魚 / 百合蒸东星斑 / 炒結頭菜乾 / 🍲懶人火鍋_康寶極湯味日式高湯包,
6,(empty box),青椒炒肉絲 - 家常料理 / 好市多牛三寶炒蒜苗 / 蒜香鮮蝦炒彩蔬 / 家常小菜-涼拌干絲 / 《料理簡單做》涼拌萵筍/菜心 / 客家鹹湯圓的“炒料”,
7,七點先放工，要十五分鐘搞得掂,十三香炒回鍋肉 / 蒜炒小白菜/快炒/5分鐘/簡單快速,
8,婆婆糖尿，醬汁唔可以甜 [chips: more_veg],好市多牛三寶炒蒜苗 / 炒櫻桃鴨胸 / 蝦仁炒毛豆 / 甜豌豆炒菇 / 涼拌櫛瓜 / 蒜炒菜心 / 薑絲高麗菜炒小黃瓜 / 白菜蛋花湯,
9,今晚想吃清蒸的，配一個燙青菜,清蒸山藥｜三鮮肉丸 / 清蒸石斑魚 / 家常小菜-涼拌干絲 / 榭黃芥菜羹｜立春節氣料理,


## 5 · Record the results and write the Excel file

Repeat the 20 requests by hand on the deployed Streamlit Cloud app and confirm each row in the
"Checked on Streamlit Cloud" column of the Excel sheet.

In [9]:
(OUT / "experiments.json").write_text(json.dumps({
    "p2_candidates": list(P2_CANDIDATES), "p2_results": p2_results, "p2_chosen": P2_CHOSEN,
    "p2_weight_curves": {m: {str(w): a for w, a in c.items()} for m, c in weight_curves.items()},
    "p2_counts": {"library": len(doc_ids), "development": len(dev), "literal": len(test), "paraphrase": len(paraphrase)},
    "p2_ranks": {k: v.tolist() for k, v in p2_ranks.items()},
    "p2_queries": [{"query": r["query"], "target": library.get(r["dish_id"]).name} for r in test],
    "p1_model": f"{HF_USER}/same-table-tagger",
    "p1_app_accuracy": p1_app_accuracy, "p1_app_seconds_per_100": 100 * p1_app_seconds / len(p1_test),
    "app_tests": app_rows, "app_accuracy": app_accuracy, "app_seconds_per_request": app_seconds / len(app_rows),
    "device": DEVICE}, ensure_ascii=False, indent=1), encoding="utf-8")
subprocess.run([sys.executable, str(ROOT / "scripts" / "export_results.py")], check=True)
print("wrote", OUT / "Experimental_results.xlsx")
print(f'''
# Streamlit Cloud -> Settings -> Secrets (Pipeline 2; these are also the defaults in same_table/config.py)
P2_MODEL_ID      = "{P2_CHOSEN}"
P2_POOLING       = "{pooling}"
P2_HYBRID_WEIGHT = "{P2_WEIGHT}"
''')

wrote /home/claude/same_table_app/artifacts/Experimental_results.xlsx
wrote /home/claude/same_table_app/artifacts/Experimental_results.xlsx

# Streamlit Cloud -> Settings -> Secrets (Pipeline 2; these are also the defaults in same_table/config.py)
P2_MODEL_ID      = "BAAI/bge-small-zh-v1.5"
P2_POOLING       = "cls"
P2_HYBRID_WEIGHT = "3.0"

